# Practice with PyTorch Forecasting, HuggingFace Datasets, and MAPIE

This notebook contains code for the experimentation with the pytorch-forecasting, datasets, and mapie Python libraries to help with the development of the project. It will consist of importing the data with datasets, constructing a temporal fusion transformer (TFT), and using MAPIE to create conformal prediction intervals. No imputation.

In [3]:
# Imports

import copy
from pathlib import Path
import warnings

import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
import numpy as np
import pandas as pd
import torch

from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import (
    optimize_hyperparameters,
)

import plotly.express as px
from scipy.stats import randint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV

from mapie.metrics.regression import (
    coverage_width_based,
    regression_coverage_score,
    regression_mean_width_score,
)
from mapie.regression import TimeSeriesRegressor
from mapie.subsample import BlockBootstrap

warnings.simplefilter("ignore")
from datasets import load_dataset

In [ ]:
# import the dataset
dataset = load_dataset(
    "Dingdong-Inc/FreshRetailNet-50K",
    cache_dir="/home/gil/projects/uncertainty_aware_planner/data",
)

{'city_id': 0, 'store_id': 0, 'management_group_id': 0, 'first_category_id': 5, 'second_category_id': 6, 'third_category_id': 65, 'product_id': 38, 'dt': '2024-03-28', 'sale_amount': 0.1, 'hours_sale': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'stock_hour6_22_cnt': 0, 'hours_stock_status': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'discount': 1.0, 'holiday_flag': 0, 'activity_flag': 0, 'precpt': 1.6999, 'avg_temperature': 15.48, 'avg_humidity': 73.54, 'avg_wind_level': 1.97}
